# Pelatihan Model YOLO11n untuk VNetra (Navigasi Tunanetra)
Notebook ini dibuat untuk berjalan di **Google Colab**.
Pastikan Anda telah mengaktifkan GPU dengan cara masuk ke menu runtime (T4 GPU).

Notebook ini akan mengeksekusi *pipeline* berikut:
1. Mengunduh dataset kustom dari Roboflow (Pothole, Tactile Paving, Drain, dll).
2. Mengunduh Subset COCO untuk mencegah *Catastrophic Forgetting*.
3. Menyatukan dataset dengan ID kelas (Class ID) yang selaras (25 Classes).
4. Melatih model YOLO11n menggunakan `ultralytics` dengan *augmentation* khusus kamera OV2640.
5. Mengekspor model menjadi `.tflite` (FP16 & INT8).

In [ ]:
import IPython
import PIL
pil_ver = PIL.__version__
print(f"Mengunci versi Pillow ke {pil_ver} untuk mencegah crash C-extension...")
IPython.get_ipython().system(f"pip install ultralytics roboflow pyyaml fiftyone Pillow=={pil_ver}")

import importlib
import site
importlib.reload(site)
importlib.invalidate_caches()

import os
import shutil
import yaml
import glob
from roboflow import Roboflow

print("Environment siap!")


## 1. Unduh Dataset Kustom dari Roboflow
Menarik semua dataset rintangan spesifik tunanetra dari Roboflow Universe.

In [ ]:
# Mengambil API Key secara otomatis dari sistem Secret Kaggle/Colab
import os
from roboflow import Roboflow

roboflow_key = "YOUR_ROBOFLOW_API_KEY_HERE"

# Deteksi jika berjalan di Kaggle
if os.path.exists("/kaggle"):
    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        roboflow_key = user_secrets.get_secret("ROBOFLOW_API_KEY")
    except Exception as e:
        print("Gagal mengambil kunci dari Kaggle Secrets:", e)
# Deteksi jika berjalan di Google Colab
elif "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ:
    try:
        from google.colab import userdata
        roboflow_key = userdata.get('ROBOFLOW_API_KEY')
    except Exception as e:
        print("Gagal mengambil kunci dari Colab Userdata:", e)

if roboflow_key == "YOUR_ROBOFLOW_API_KEY_HERE" or not roboflow_key:
    raise ValueError("\n\n[ERROR FATAL] API Key Roboflow belum diatur!\nSilakan buat variabel rahasia dengan nama 'ROBOFLOW_API_KEY' di menu Add-ons -> Secrets (Kaggle) atau logo Kunci (Colab).\n")

rf = Roboflow(api_key=roboflow_key)
dataset_pothole = rf.workspace("yeeun-kim-fyvoj").project("pothole-vhmow").version(18).download("yolov11")

# 2. Tactile Paving Dataset
dataset_tactile = rf.workspace("raihan-aria").project("paving-tactile-detection").version(4).download("yolov11")

# 3. Open Drain Dataset
dataset_drain = rf.workspace("chaitanya-kharche").project("drain-overflow").version(2).download("yolov11")

# 4. Puddle Dataset
dataset_puddle = rf.workspace("ambitious-jda7x").project("puddle-zlrsu").version(2).download("yolov11")
dataset_puddle2 = rf.workspace("tt-xsaer").project("fvs").version(4).download("yolov11")

# 5. Pole Dataset
dataset_pole = rf.workspace("ghost-gsj7h").project("utility-pole-aka9k").version(3).download("yolov11")

# 6. Hanging Branch Dataset
dataset_branch = rf.workspace("utem").project("branch-7qne7").version(2).download("yolov11")
dataset_branch3 = rf.workspace("ahmdirfnz").project("branch").version(5).download("yolov11")

# 7. Stairs Dataset
dataset_stairs = rf.workspace("jatin-sne2e").project("stairs-zqsvn").version(2).download("yolov11")

# 8. Curb Ramp Dataset (Undakan Trotoar)
dataset_curb = rf.workspace("curbstone").project("curb-zjoev").version(1).download("yolov11")
dataset_curb2 = rf.workspace("rad-khosh-kqqsc").project("curb-sizeall").version(3).download("yolov11")
dataset_curb3 = rf.workspace("bens-cv-projects").project("mebot-curb-detection").version(2).download("yolov11")

# 9. Tree
dataset_tree = rf.workspace("tree-nqhzs").project("tree-hmf5d").version(1).download("yolov11")

# 10. Crosswalk
dataset_crosswalk = rf.workspace("wqwdas").project("crosswalk-1elwe").version(1).download("yolov11")

# 11. Fence
dataset_fence = rf.workspace("ayoub-9grd0").project("fence-detection-bkrx1").version(1).download("yolov11")


## 2. Unduh Dataset Bawaan COCO (Anti-Catastrophic Forgetting)
Mencegah YOLO melupakan wujud manusia, mobil, atau motor karena tertimpa data Roboflow di atas. Kita mengambil 3000 gambar subset dari COCO-2017 menggunakan `fiftyone`.

In [ ]:
import os
import urllib.request
import zipfile
import json
import concurrent.futures

coco_classes = [
    "person", "bicycle", "car", "motorcycle", "bus", "truck", "train", 
    "stop sign", "bench", "chair", "potted plant", "dog", "cat"
]

print("=== Menggunakan FiftyOne ===")
import fiftyone as fo
import fiftyone.zoo as foz

print("Mulai mengunduh subset COCO-2017 (Estimasi waktu: 3-5 menit)...")
coco_dataset = foz.load_zoo_dataset(
    "coco-2017", split="train", label_types=["detections"],
    classes=coco_classes, max_samples=3000
)

coco_export_dir = "/content/coco_subset"
coco_dataset.export(
    export_dir=coco_export_dir, dataset_type=fo.types.YOLOv5Dataset,
    label_field="ground_truth", classes=coco_classes
)
print(f"Selesai! Subset COCO diekstrak ke {coco_export_dir}\n")


## 3. Penggabungan (Merging) Seluruh Dataset
Menyatukan seluruh dataset (10 Roboflow + 1 COCO) ke dalam folder `vnetra_master_dataset` sambil merekayasa ID Kelas mereka agar berurutan (0-24) secara konsisten.

In [ ]:
import glob
master_dir = "/content/vnetra_master_dataset"

    copied_count = 0
for split in ['train', 'valid', 'test']:
    os.makedirs(f"{master_dir}/{split}/images", exist_ok=True)
    os.makedirs(f"{master_dir}/{split}/labels", exist_ok=True)

# Definisi urutan Master Class VNetra (COCO + Custom)
master_classes = [
    'person', 'bicycle', 'car', 'motorcycle', 'bus', 'truck', 'train', 
    'stop sign', 'bench', 'chair', 'potted plant', 'dog', 'cat',
    'pothole', 'open_drain', 'puddle', 'pole', 
    'hanging_branch', 'tactile_paving_straight', 'tactile_paving_turn', 
    'tactile_paving_3way', 'tactile_paving_4way', 'tactile_paving_stop', 
    'stairs_up', 'stairs_down', 'curb', 'crosswalk', 'tree', 'fence'
]
master_class_to_id = {name: i for i, name in enumerate(master_classes)}

def merge_dataset(source_path, class_mapping, is_coco=False, max_samples=None):
    if not os.path.exists(source_path): return
    
    # Ambil original classes dari data.yaml jika bukan coco
    if not is_coco:
        yaml_path = os.path.join(source_path, 'data.yaml')
        if not os.path.exists(yaml_path): return
        with open(yaml_path, 'r') as f:
            data_info = yaml.safe_load(f)
            original_classes = data_info['names']
            if isinstance(original_classes, dict):
                original_classes = [original_classes[i] for i in range(len(original_classes))]
    else:
        original_classes = coco_classes
        
    copied_count = 0
    for split in ['train', 'valid', 'test']:
        # Struktur Roboflow: train/images
        # Struktur FiftyOne: images/train (dan split val disebut 'val')
        fo_split = 'val' if split == 'valid' else split
        
        # Jika COCO dan FiftyOne menaruhnya di 'val', paksa pindah ke 'train'
        if is_coco and split == 'train':
            if os.path.exists(f"{source_path}/images/val"): fo_split = 'val'
        elif is_coco and split != 'train':
            continue # COCO kita fokuskan di Train, biarkan Rebalancer yang membagi
        img_dir_rf = f"{source_path}/{split}/images"
        img_dir_fo = f"{source_path}/images/{fo_split}"
        
        if os.path.exists(img_dir_rf):
            img_dir = img_dir_rf
            lbl_dir_base = f"{source_path}/{split}/labels"
        elif os.path.exists(img_dir_fo):
            img_dir = img_dir_fo
            lbl_dir_base = f"{source_path}/labels/{fo_split}"
        else:
            continue
            
        import random
        all_images = glob.glob(f"{img_dir}/*")
        random.shuffle(all_images)
        for img_path in all_images:
            if max_samples is not None and copied_count >= max_samples:
                return
            file_name = os.path.basename(img_path)
            lbl_name = file_name.rsplit('.', 1)[0] + '.txt'
            lbl_path = f"{lbl_dir_base}/{lbl_name}"
            
            if not os.path.exists(lbl_path): continue
                
            new_labels = []
            valid_objects = 0
            with open(lbl_path, 'r') as f:
                lines = f.readlines()
            
            for line in lines:
                parts = line.strip().split()
                if len(parts) < 5: continue
                
                orig_id = int(parts[0])
                if orig_id >= len(original_classes): continue
                
                class_name = str(original_classes[orig_id]).lower()
                
                # Case-insensitive mapping
                mapped_master_class = None
                for key, val in class_mapping.items():
                    if key.lower() == class_name:
                        mapped_master_class = val
                        break
                        
                if mapped_master_class:
                    new_id = master_class_to_id[mapped_master_class]
                    new_labels.append(f"{new_id} {' '.join(parts[1:])}\n")
                    valid_objects += 1
            
            if valid_objects > 0:
                prefix = source_path.split('/')[-1]
                new_img_name = f"{prefix}_{file_name}"
                new_lbl_name = f"{prefix}_{lbl_name}"
                
                shutil.copy(img_path, f"{master_dir}/{split}/images/{new_img_name}")
                with open(f"{master_dir}/{split}/labels/{new_lbl_name}", 'w') as f:
                    f.writelines(new_labels)
                    copied_count += 1

print("Memproses COCO Subset (Mencegah Catastrophic Forgetting)...")
merge_dataset(coco_export_dir, {c: c for c in coco_classes}, is_coco=True)

print("Memproses Pothole Dataset...")
merge_dataset(dataset_pothole.location, {"pothole": "pothole"})

print("Memproses Tactile Paving Dataset...")
merge_dataset(dataset_tactile.location, {"go": "tactile_paving_straight", "1": "tactile_paving_straight", "0": "tactile_paving_straight", "straight": "tactile_paving_straight", "2": "tactile_paving_turn", "3": "tactile_paving_3way", "4": "tactile_paving_4way", "stop": "tactile_paving_stop"})

print("Memproses Open Drain Dataset...")
merge_dataset(dataset_drain.location, {"open drainage-not overflowing-": "open_drain", "open drainage-overflowing-": "open_drain", "drainage overflow-repair-": "open_drain", "open manhole-not overflowing-": "open_drain"})

print("Memproses Puddle Dataset...")
merge_dataset(dataset_puddle.location, {"puddle": "puddle"})
merge_dataset(dataset_puddle2.location, {"puddle": "puddle"})

print("Memproses Pole Dataset...")
merge_dataset(dataset_pole.location, {"pole": "pole", "pole_including_insulator": "pole"})

print("Memproses Hanging Branch Dataset...")
merge_dataset(dataset_branch.location, {"branch": "hanging_branch", "branches": "hanging_branch", "0": "hanging_branch"})

print("Memproses Stairs Dataset...")
merge_dataset(dataset_stairs.location, {"downstair": "stairs_down", "upstair": "stairs_up", "stairs_up": "stairs_up", "stairs_down": "stairs_down"})

print("Memproses Curb Ramp Dataset...")
merge_dataset(dataset_curb.location, {"curb": "curb"})

print("Memproses Tree Dataset...")
merge_dataset(dataset_tree.location, {"tree": "tree"}, max_samples=2500)

print("Memproses Tambahan Dataset Branch...")
merge_dataset(dataset_branch3.location, {"branches": "hanging_branch"})

print("Memproses Tambahan Dataset Curb...")
merge_dataset(dataset_curb2.location, {"curb": "curb"})
print("Memproses Tambahan Dataset Curb 3...")
try:
    with open(f"{dataset_curb3.location}/data.yaml", "r") as f:
        c3_classes = yaml.safe_load(f)["names"]
    if isinstance(c3_classes, dict): c3_classes = [c3_classes[i] for i in range(len(c3_classes))]
    c3_map = {str(c): "curb" for c in c3_classes if "curb" in str(c).lower()}
    merge_dataset(dataset_curb3.location, c3_map)
except:
    pass

print("Memproses Fence Dataset...")
merge_dataset(dataset_fence.location, {"fences": "fence"})

print("Memproses Crosswalk Dataset...")
try:
    with open(f"{dataset_crosswalk.location}/data.yaml", 'r') as f:
        cw_classes = yaml.safe_load(f)['names']
    if isinstance(cw_classes, dict): cw_classes = [cw_classes[i] for i in range(len(cw_classes))]
    cw_map = {str(c): "crosswalk" for c in cw_classes if str(c).lower() != "null"}
    merge_dataset(dataset_crosswalk.location, cw_map)
except:
    pass

yaml_content = {
    "path": master_dir,
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": len(master_classes),
    "names": master_classes
}

with open(f"{master_dir}/data.yaml", 'w') as f:
    yaml.dump(yaml_content, f, sort_keys=False)



### 3.1 Jaring Pengaman Rebalancing Data
Terkadang kreator dataset di Roboflow lupa membagi data *Validation*, atau COCO subset seluruhnya masuk ke *Train*. Skrip ini memindahkan gambar secara otomatis dari *Train* ke *Valid* agar **tidak ada satu pun kelas yang memiliki soal ujian di bawah 50 gambar**.

In [ ]:
import os
import random
import shutil

print("=== MEMASTIKAN SEMUA KELAS MUNCUL DI VALIDATION SET ===")
train_img_dir = 'vnetra_master_dataset/train/images'
train_lbl_dir = 'vnetra_master_dataset/train/labels'
valid_img_dir = 'vnetra_master_dataset/valid/images'
valid_lbl_dir = 'vnetra_master_dataset/valid/labels'

os.makedirs(valid_img_dir, exist_ok=True)
os.makedirs(valid_lbl_dir, exist_ok=True)

valid_class_counts = {i: 0 for i in range(len(master_classes))}
valid_labels = [f for f in os.listdir(valid_lbl_dir) if f.endswith('.txt')]

for lbl_file in valid_labels:
    with open(os.path.join(valid_lbl_dir, lbl_file), 'r') as f:
        for line in f:
            parts = line.strip().split()
            if parts:
                cls_id = int(parts[0])
                if cls_id in valid_class_counts:
                    valid_class_counts[cls_id] += 1

target_min_instances = 50
classes_to_boost = [cls_id for cls_id, count in valid_class_counts.items() if count < target_min_instances]

if not classes_to_boost:
    print(f"\nSemua 29 kelas sudah memiliki minimal {target_min_instances} instans di Valid Set! Aman.")
else:
    print(f"\nAda kelas yang kosong/kurang data di Valid Set: {classes_to_boost}")
    print("Meminjam gambar secara acak dari folder Train...")
    
    train_labels = [f for f in os.listdir(train_lbl_dir) if f.endswith('.txt')]
    random.shuffle(train_labels)
    
    moved_images = 0
    for lbl_file in train_labels:
        if not classes_to_boost:
            break
            
        src_lbl = os.path.join(train_lbl_dir, lbl_file)
        contains_needed_class = False
        with open(src_lbl, 'r') as f:
            lines = f.readlines()
            
        for line in lines:
            parts = line.strip().split()
            if parts:
                cls_id = int(parts[0])
                if cls_id in classes_to_boost:
                    contains_needed_class = True
                    break
                    
        if contains_needed_class:
            dst_lbl = os.path.join(valid_lbl_dir, lbl_file)
            img_file_base = os.path.splitext(lbl_file)[0]
            
            src_img = None
            dst_img = None
            for ext in ['.jpg', '.jpeg', '.png']:
                temp_src = os.path.join(train_img_dir, img_file_base + ext)
                if os.path.exists(temp_src):
                    src_img = temp_src
                    dst_img = os.path.join(valid_img_dir, img_file_base + ext)
                    break
            
            if src_img and os.path.exists(src_img):
                shutil.move(src_img, dst_img)
                shutil.move(src_lbl, dst_lbl)
                moved_images += 1
                
                for line in lines:
                    parts = line.strip().split()
                    if parts:
                        c_id = int(parts[0])
                        if c_id in valid_class_counts:
                            valid_class_counts[c_id] += 1
                            
                classes_to_boost = [c for c, count in valid_class_counts.items() if count < target_min_instances]

    print(f"\nBerhasil memindahkan {moved_images} gambar + label dari Train ke Valid!")



### 3.2 Laporan Akhir Proporsi Dataset
Menghitung dan menampilkan tabel rekapitulasi jumlah *Train*, *Valid*, dan *Test* untuk memverifikasi apakah semua 29 kelas sudah berimbang sebelum di- *training* oleh algoritma YOLO.

In [ ]:
import os
import pandas as pd

print("\n=======================================================")
print("      LAPORAN AKHIR DISTRIBUSI DATASET VNETRA")
print("=======================================================")

def count_images(directory):
    if not os.path.exists(directory): return 0
    return len([f for f in os.listdir(directory) if f.endswith(('.jpg', '.jpeg', '.png'))])

train_count = count_images('vnetra_master_dataset/train/images')
valid_count = count_images('vnetra_master_dataset/valid/images')
test_count  = count_images('vnetra_master_dataset/test/images')
total_images = train_count + valid_count + test_count

if total_images > 0:
    print(f"Total Lembar Gambar (Images) Keseluruhan : {total_images} lembar")
    print(f"- Data Latih (Train)     : {train_count} lembar ({train_count/total_lembar*100:.2f}%)")
    print(f"- Data Ujian (Valid)     : {valid_count} lembar ({valid_count/total_lembar*100:.2f}%)")
    print(f"- Data Buta (Test)       : {test_count} lembar ({test_count/total_lembar*100:.2f}%)")
else:
    print("Belum ada gambar yang diproses.")

print('\n[DEBUG] Mengecek letak folder COCO asli dari FiftyOne:')
import glob
coco_base = '/content/vnetra_coco_subset'
print(f'Train images: {len(glob.glob(coco_base + "/images/train/*"))}')
print(f'Val images: {len(glob.glob(coco_base + "/images/val/*"))}')
print(f'Master Train: {len(glob.glob(master_dir + "/train/images/*coco*"))}')
print(f'Master Valid: {len(glob.glob(master_dir + "/valid/images/*coco*"))}')
print("\n--- RINCIAN INSTANS (JUMLAH OBJEK / BOUNDING BOX) PER KELAS ---")
print("Catatan: Tabel di bawah ini murni menghitung jumlah instans objek di dalam gambar, BUKAN jumlah lembar gambar.")
def count_instances_per_class(label_dir, num_classes):
    counts = {i: 0 for i in range(num_classes)}
    if not os.path.exists(label_dir): return counts
    
    for lbl_file in os.listdir(label_dir):
        if not lbl_file.endswith('.txt'): continue
        with open(os.path.join(label_dir, lbl_file), 'r') as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    cls_id = int(parts[0])
                    if cls_id in counts:
                        counts[cls_id] += 1
    return counts

train_cls = count_instances_per_class('vnetra_master_dataset/train/labels', len(master_classes))
valid_cls = count_instances_per_class('vnetra_master_dataset/valid/labels', len(master_classes))
test_cls  = count_instances_per_class('vnetra_master_dataset/test/labels', len(master_classes))

data_report = []
for i, cls_name in enumerate(master_classes):
    data_report.append({
        'ID': i,
        'Kelas': cls_name,
        'Train': train_cls[i],
        'Valid': valid_cls[i],
        'Test': test_cls[i]
    })

df_report = pd.DataFrame(data_report)
display(df_report)
print("\n[INFO] Semua kelas di atas telah lolos jaring pengaman (Minimal 50 instans di Valid Set). Aman untuk lanjut training!")



In [ ]:
import os
# Simpan Dataset Hasil Penggabungan
os.system('zip -r /content/drive/MyDrive/vnetra_master_dataset.zip /content/vnetra_master_dataset > /dev/null')
print('Dataset berhasil di-zip dan disimpan ke Google Drive!')


## 4. Training YOLO11n dengan Augmentasi OV2640
Melatih model YOLOv11 versi Nano dengan augmentasi hyperparameter yang dikhususkan untuk **motion blur, rotasi sudut jalan, dan fluktuasi pencahayaan** yang sering ditemui pada kamera OV2640 di perangkat wearable.

In [ ]:
from ultralytics import YOLO

# Load pre-trained model (YOLO11 Nano)
model = YOLO('yolo11n.pt')

# Mulai proses training
results = model.train(
    data=f"{master_dir}/data.yaml",
    epochs=300,            # Diatur tinggi karena ada Early Stopping
    patience=50,           # Otomatis berhenti jika tidak ada peningkatan mAP selama 50 epoch
    imgsz=640,             # Sesuai dengan resolusi VGA kamera ESP32-S3 (OV2640)
    batch=32,              # Batch ukuran 32 untuk 1 GPU T4 (mencegah Out of Memory)
    device=0,              # Memanfaatkan 1x GPU T4 di Colab agar tidak error
    seed=42,               # Menjaga hasil tetap bisa direproduksi (untuk Skripsi)
    project='vnetra_training',
    name='yolo11n_custom',
    
    # --- HYPERPARAMETER AUGMENTASI UNTUK KAMERA OV2640 ---
    mosaic=1.0,      # Menggabungkan 4 gambar jadi 1 untuk ketangguhan deteksi terpotong
    degrees=15.0,    # Toleransi rotasi untuk simulasi goyangan kamera saat berjalan
    hsv_h=0.015,     # Simulasi fluktuasi cahaya matahari/lampu jalan
    hsv_s=0.7,       # Simulasi warna pudar kamera analog
    hsv_v=0.4,       # Simulasi kontras rendah / kondisi backlight matahari
)

In [ ]:
import shutil
import os
shutil.copy('vnetra_training/yolo11n_custom/weights/best.pt', '/content/drive/MyDrive/best_yolo11n.pt')
print('Model Asli (.pt) berhasil disimpan ke Google Drive!')


## 5. Export ke TensorFlow Lite (TFLite)
Mengekspor bobot model menjadi format `.tflite` dalam dua bentuk kuantisasi:
1. **FP16** (Half Precision) -> Sangat efisien dan kompatibel untuk *GPU Delegation* di Android.
2. **INT8** (Full Integer) -> Wajib untuk akselerator *NPU / NNAPI* yang membutuhkan model super ringan.

In [ ]:
print("Mengekspor model ke FP16...")
# 1. Export ke TFLite (FP16) - Optimal untuk GPU Mobile (Proses cepat)
export_fp16 = model.export(format="tflite", half=True, optimize=True)
print("===========================================================")
print("Export FP16 Selesai! Lokasi file TFLite:")
print("FP16:", export_fp16)
print("===========================================================")


In [ ]:
import shutil
import os
shutil.copy(export_fp16, '/content/drive/MyDrive/best_fp16.tflite')
print('Model FP16 berhasil disimpan ke Google Drive!')


### Kuantisasi INT8 (Opsional / Terpisah)
Proses ini memakan waktu sangat lama (10-25 menit) di GPU biasa karena harus melakukan kalibrasi dataset. Eksekusi *cell* di bawah ini hanya jika Anda membutuhkan bobot ekstra ringan untuk *Neural Processing Unit* (NPU/NNAPI).

**⚡ TPU Acceleration Trick (Sangat Disarankan):**
Jika Anda ingin mempercepat kalibrasi ini secara drastis menggunakan puluhan *core* CPU raksasa dari mesin TPU Colab:
1. **Putuskan** mesin GPU saat ini: `Runtime` -> `Disconnect and delete runtime`.
2. **Ganti** ke mesin TPU: `Runtime` -> `Change runtime type` -> `TPU`.
3. **Sambungkan** dan jalankan *cell* penginstalan *library* serta *Mount Drive* di paling atas.
4. Lewati semua proses *download* dan *training*. Langsung jalankan *cell TPU Preparation* di bawah ini, dilanjutkan dengan ekspor INT8.

In [ ]:
import os
import torch
master_dir = "/content/vnetra_master_dataset"

if not torch.cuda.is_available():
    print("\u26a1 Mendeteksi mesin non-GPU (Berjalan di mesin TPU/CPU).")
    print("Menyiapkan dataset dan model dari Google Drive...")
    
    # 1. Ekstrak ulang dataset dari Drive
    if not os.path.exists(master_dir):
        os.system('unzip -q /content/drive/MyDrive/vnetra_master_dataset.zip -d /content/')
        print("[V] Dataset berhasil diekstrak ulang dari Drive!")
    else:
        print("[V] Dataset sudah siap.")
        
    # 2. Muat ulang model dari Drive
    try:
        model
        print("[V] Model sudah ada di memori.")
    except NameError:
        from ultralytics import YOLO
        model = YOLO('/content/drive/MyDrive/best_yolo11n.pt')
        print("[V] Model berhasil dimuat dari Google Drive!")
else:
    print("\ud83d\udd25 Mendeteksi mesin GPU aktif. Anda masih berada di sesi Training asli.")


In [ ]:
print("Mengekspor model ke INT8 secara murni menggunakan CPU...")
print("Silakan tunggu 10-25 menit (proses kalibrasi dataset), layar mungkin terlihat stuck...")
# 2. Export ke TFLite (INT8) - Wajib untuk NPU / NNAPI
export_int8 = model.export(format="tflite", int8=True, data=f"{master_dir}/data.yaml", optimize=True, device='cpu')
print("===========================================================")
print("Export INT8 Selesai! Lokasi file TFLite:")
print("INT8:", export_int8)
print("===========================================================")


In [ ]:
import shutil
import os
shutil.copy(export_int8, '/content/drive/MyDrive/best_int8.tflite')
print('Model INT8 berhasil disimpan ke Google Drive!')


## 6. Validasi Kuantisasi (Benchmarking Skripsi)
Menguji kembali model pada Validation Set untuk melihat seberapa jauh penurunan akurasi (mAP) akibat proses kompresi FP16 dan INT8 dibanding model aslinya.

In [ ]:
!pip install tensorflow
import gc
gc.collect()

print("\n=== EVALUASI MODEL ASLI (.pt) ===")
val_pt = model.val(data=f"{master_dir}/data.yaml")
map_pt = val_pt.box.map50

print("\n=========================================")
print("mAP@50 (Akurasi):")
print(f"Original (.pt)   : {map_pt:.4f}")

In [ ]:
print("\n=== EVALUASI MODEL FP16 (.tflite) ===")
model_fp16 = YOLO(export_fp16, task='detect')
val_fp16 = model_fp16.val(data=f"{master_dir}/data.yaml")
map_fp16 = val_fp16.box.map50

print("\n=========================================")
print("mAP@50 (Akurasi):")
print(f"FP16 (.tflite)   : {map_fp16:.4f}")

In [ ]:
print("\n=== EVALUASI MODEL INT8 (.tflite) ===")
model_int8 = YOLO(export_int8, task='detect')
val_int8 = model_int8.val(data=f"{master_dir}/data.yaml")
map_int8 = val_int8.box.map50

print("\n=========================================")
print("mAP@50 (Akurasi):")
print(f"INT8 (.tflite)   : {map_int8:.4f}")

In [ ]:
print("\n=========================================")
print("KESIMPULAN PERBANDINGAN mAP@50 (Akurasi):")
print(f"Original (.pt)   : {map_pt:.4f}")
print(f"FP16 (.tflite)   : {map_fp16:.4f}")
print(f"INT8 (.tflite)   : {map_int8:.4f}")
print("=========================================")

## 7. Pengujian Visualisasi Langsung (Predict)
Mengambil satu gambar tes secara acak dan menampilkan prediksi kotak deteksi dari model asli (.pt) vs model terkompresi (.tflite) agar Anda bisa meletakkannya di Laporan Skripsi.

In [ ]:
import random
import matplotlib.pyplot as plt
import cv2

# Pilih satu gambar acak dari dataset test (atau valid jika test tidak ada)
test_images = glob.glob(f"{master_dir}/valid/images/*.jpg")
if test_images:
    test_img = random.choice(test_images)
    print(f"Menguji gambar: {test_img}")
    
    # Prediksi pakai model Asli
    res_pt = model.predict(source=test_img, imgsz=640)
    img_pt = res_pt[0].plot()
    
    # Prediksi pakai model INT8
    res_int8 = model_int8.predict(source=test_img, imgsz=640)
    img_int8 = res_int8[0].plot()
    
    # Tampilkan perbandingan
    fig, ax = plt.subplots(1, 2, figsize=(15, 7))
    ax[0].imshow(cv2.cvtColor(img_pt, cv2.COLOR_BGR2RGB))
    ax[0].set_title("Prediksi Model Asli (.pt)")
    ax[0].axis("off")
    
    ax[1].imshow(cv2.cvtColor(img_int8, cv2.COLOR_BGR2RGB))
    ax[1].set_title("Prediksi Model INT8 (.tflite)")
    ax[1].axis("off")
    
    plt.tight_layout()
    plt.show()
else:
    print("Tidak ada gambar di folder valid untuk diprediksi.")


## 8. Visualisasi Grafik Hasil Training
Menampilkan grafik metrik akurasi (*mAP*, *Loss*) dan *Confusion Matrix* yang telah digenerasi oleh YOLO menggunakan `matplotlib` untuk keperluan laporan skripsi.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Image

base_path = '/content/vnetra_training/yolo11n_custom/'
results_path = os.path.join(base_path, 'results.csv')

print('=== 1. CUSTOM TRAINING DASHBOARD (SEABORN) ===')
if os.path.exists(results_path):
    df = pd.read_csv(results_path)
    df.columns = df.columns.str.strip()  # Bersihkan spasi berlebih di nama kolom
    
    sns.set_theme(style='whitegrid', palette='deep')
    fig, axes = plt.subplots(2, 2, figsize=(20, 14))
    fig.suptitle('VNetra - YOLO11n Training Performance Dashboard', fontsize=26, fontweight='bold', y=0.96)
    
    # 1. Box Loss Convergence
    sns.lineplot(data=df, x='epoch', y='train/box_loss', ax=axes[0,0], label='Train Box Loss', linewidth=3, color='#1f77b4')
    sns.lineplot(data=df, x='epoch', y='val/box_loss', ax=axes[0,0], label='Val Box Loss', linewidth=3, color='#ff7f0e', linestyle='--')
    axes[0,0].set_title('Box Loss Convergence', fontsize=18, fontweight='bold')
    axes[0,0].set_xlabel('Epoch', fontsize=14)
    axes[0,0].set_ylabel('Loss', fontsize=14)
    axes[0,0].legend(fontsize=12, frameon=True, shadow=True)
    
    # 2. Class Loss Convergence
    sns.lineplot(data=df, x='epoch', y='train/cls_loss', ax=axes[0,1], label='Train Class Loss', linewidth=3, color='#2ca02c')
    sns.lineplot(data=df, x='epoch', y='val/cls_loss', ax=axes[0,1], label='Val Class Loss', linewidth=3, color='#d62728', linestyle='--')
    axes[0,1].set_title('Classification Loss Convergence', fontsize=18, fontweight='bold')
    axes[0,1].set_xlabel('Epoch', fontsize=14)
    axes[0,1].set_ylabel('Loss', fontsize=14)
    axes[0,1].legend(fontsize=12, frameon=True, shadow=True)
    
    # 3. mAP Score Evolution
    sns.lineplot(data=df, x='epoch', y='metrics/mAP50(B)', ax=axes[1,0], label='mAP@50', linewidth=3, color='#9467bd')
    sns.lineplot(data=df, x='epoch', y='metrics/mAP50-95(B)', ax=axes[1,0], label='mAP@50-95', linewidth=3, color='#8c564b', linestyle='-.')
    axes[1,0].set_title('Mean Average Precision (mAP)', fontsize=18, fontweight='bold')
    axes[1,0].set_xlabel('Epoch', fontsize=14)
    axes[1,0].set_ylabel('Score', fontsize=14)
    axes[1,0].legend(fontsize=12, frameon=True, shadow=True)
    
    # 4. Precision & Recall
    sns.lineplot(data=df, x='epoch', y='metrics/precision(B)', ax=axes[1,1], label='Precision', linewidth=3, color='#e377c2')
    sns.lineplot(data=df, x='epoch', y='metrics/recall(B)', ax=axes[1,1], label='Recall', linewidth=3, color='#17becf', linestyle=':')
    axes[1,1].set_title('Precision & Recall Trends', fontsize=18, fontweight='bold')
    axes[1,1].set_xlabel('Epoch', fontsize=14)
    axes[1,1].set_ylabel('Score', fontsize=14)
    axes[1,1].legend(fontsize=12, frameon=True, shadow=True)
    
    plt.tight_layout(rect=[0, 0, 1, 0.94])
    plt.show()
else:
    print('File results.csv belum ditemukan.')

def display_result(image_path, width=None):
    if os.path.exists(image_path):
        if width:
            display(Image(filename=image_path, width=width))
        else:
            display(Image(filename=image_path))
    else:
        print(f'\u26a0\ufe0f Gambar tidak ditemukan: {os.path.basename(image_path)}')
        print('   (Gambar ini baru akan digenerate oleh YOLO di detik terakhir setelah epoch 50 selesai 100%)')

print('\n=== 2. CONFUSION MATRIX PROFESIONAL ===')
display_result(os.path.join(base_path, 'confusion_matrix_normalized.png'), width=1200)

print('\n=== 3. KURVA F1-SCORE (Confidence Thresholding) ===')
display_result(os.path.join(base_path, 'F1_curve.png'), width=1200)

print('\n=== 4. VISUALISASI AUGMENTASI MOSAIC PADA DATA TRAINING ===')
display_result(os.path.join(base_path, 'train_batch0.jpg'), width=1200)

print('\n=== 5. SAMPEL PREDIKSI PADA VALIDATION SET ===')
display_result(os.path.join(base_path, 'val_batch0_pred.jpg'), width=1200)

